# 02 — Model evaluation

Rolling-origin results for all tiers. Numbers are produced by
`python -m src.run_experiments`; this notebook reads the cache.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import plotly.graph_objects as go
from src import config

pd.set_option("display.width", 200)
summary = pd.read_csv(config.RESULTS_DIR / "summary.csv", index_col=0)
curve   = pd.read_csv(config.RESULTS_DIR / "error_by_horizon.csv")
meta    = json.loads((config.RESULTS_DIR / "run_metadata.json").read_text())
fc      = pd.read_parquet(config.RESULTS_DIR / "forecasts.parquet")
print(f"{meta['n_test_origins']:,} test origins, stride {meta['origin_stride_hours']}h")
print(f"MASE scale {meta['mase_scale']:,.1f} MW (in-sample {meta['stronger_baseline']})")

## Headline results

Improvement is quoted against the **stronger** of the two seasonal naive baselines.

In [ ]:
summary.round(3)

## Error by forecast horizon

A single averaged MAE hides how skill decays across the 24 hours.

In [ ]:
fig = go.Figure()
for name, d in curve.groupby("model"):
    fig.add_trace(go.Scatter(x=d["h"], y=d["MAE"], name=name, mode="lines+markers"))
fig.update_layout(template="plotly_white", height=460,
                  xaxis_title="Hours ahead", yaxis_title="MAE (MW)",
                  title="Error by horizon", hovermode="x unified")
fig

## Interval calibration

An 80% interval should contain the actual about 80% of the time — not always. Coverage below nominal means overconfident intervals.

In [ ]:
cols = [c for c in ("PICP_80", "interval_width_MW", "pinball_q10", "pinball_q50", "pinball_q90")
        if c in summary.columns]
summary[cols].dropna(how="all").round(4)

## A sample forecast window

In [ ]:
model = "timesfm" if "timesfm" in set(fc["model"]) else sorted(fc["model"])[0]
sub = fc[fc["model"] == model]
origin = sorted(sub["origin"].unique())[len(sub["origin"].unique()) // 2]
w = sub[sub["origin"] == origin].sort_values("h")

fig = go.Figure()
if "q0.1" in w.columns and w["q0.1"].notna().any():
    fig.add_trace(go.Scatter(x=w["target_time"], y=w["q0.9"], line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=w["target_time"], y=w["q0.1"], fill="tonexty", name="P10-P90",
                             fillcolor="rgba(99,110,250,0.22)", line=dict(width=0)))
fig.add_trace(go.Scatter(x=w["target_time"], y=w["actual"], name="Actual",
                         line=dict(color="#111", width=2.5)))
fig.add_trace(go.Scatter(x=w["target_time"], y=w["prediction"], name=model,
                         line=dict(color="#EF553B", width=2.5, dash="dash")))
fig.update_layout(template="plotly_white", height=440, yaxis_title="Demand (MW)",
                  title=f"{model} — origin {origin}", hovermode="x unified")
fig